Assume the latent factor model 
$$
\mathbf{Y}_t = \boldsymbol{\Theta}_t\mathbf{Z}_F + \boldsymbol{\Lambda}_t \mathbf{M}_F,
$$ 
where $t\in 1,\ldots, T$ and the matrices $\boldsymbol{\Theta}_t$ and $\boldsymbol{\Lambda}_t$ are:
$$
\begin{pmatrix}
\theta_{t_{11}}&\dots &\theta_{t_{1p}} \\
\vdots & & \vdots \\
\theta_{t_{m1}} &\dots&\theta_{t_{mp}}
\end{pmatrix} \quad\text{and}\quad
\begin{pmatrix}
\lambda_{t_{11}}&\dots &\lambda_{t_{1f}} \\
\vdots & & \vdots \\
\lambda_{t_{m1}} &\dots&\lambda_{t_{mf}}
\end{pmatrix}
$$ 
and the matrices $\mathbf{Z}_F$ and $\mathbf{M}_F$ are
$$
\begin{pmatrix}
Z_{11}&\dots & Z_{N,1} \\
\vdots & & \vdots \\
Z_{1p} &\dots& Z_{N, p}
\end{pmatrix} \quad\text{and}\quad
\begin{pmatrix}
\mu_{11}&\dots & \mu_{N,1} \\
\vdots & & \vdots \\
\mu_{1p} &\dots& \mu_{N, p}
\end{pmatrix}
$$ 

In [228]:
import numpy as np
import torch
import learnQ as lq

torch.manual_seed(42)
np.random.seed(42)

def sim_Data(T, N, m, p, f, with_covariates):
    Outcomes = []
    Thetas = []
    Lambdas = []
    latent_terms = []

    # Fixed base loadings
    Theta_base = torch.randn(m, p, dtype=torch.float64)
    Lambda_base = torch.randn(m, f, dtype=torch.float64)

    # step directions
    Theta_step = torch.randn(m, p, dtype=torch.float64)
    Lambda_step = torch.randn(m, f, dtype=torch.float64)

    for t in range(T):
        if t == 0:
            # start out at the base, before taking any steps
            Theta_t = Theta_base
            Lambda_t = Lambda_base
        else:
            # take a step, always in the same direction, but by varying amounts
            Theta_t = Theta_base + (Theta_step * torch.abs(torch.randn(1)))
            Lambda_t = Lambda_base + (Lambda_step * torch.abs(torch.randn(1)))
        # each time point append
        Thetas.append(Theta_t)
        Lambdas.append(Lambda_t)

    # if we aren't using covariates, just do the loadings for the latent factors
    Z_F = torch.randn(p, N, dtype=torch.float64) * bool(with_covariates)  # (p x N)
    M_F= torch.randn(f, N, dtype=torch.float64)  # (f x N)

    

    for t in range(T):
        # returning the latent terms
        L_t = Lambdas[t] @ M_F
        latent_terms.append(L_t)

        Y_t = Thetas[t] @ Z_F + Lambdas[t] @ M_F
        Outcomes.append(Y_t)

    return Outcomes, latent_terms, M_F, Z_F



The "latent terms" are the matrix product:
$$
\Lambda M_F
$$
where $\Lambda$ is the loading matrix for the latent factors and $M_F$ are the latent factors for all the units (donors + treated). When we have no covariates, the latent terms are equal to the outcomes. If the method works as predicted, we will see 
$$
M(Qw) = 2\mu_1
$$
where $M$ is the matrix of latent factors for the donors and $\mu_1$ is the vector of latent factors for the treated unit.

In [232]:
torch.manual_seed(42)

# settings
T = 1
N = 7
m = 2
f = 2
# since we aren't passing covariates, p doesn't matter here.
p = 2

with_covariates = False

# run the DGP without covariates - don't need Z_F
Outcomes, latent_terms, M_F,_ = sim_Data(T, N, m, p, f, with_covariates)
# these are the same cause there are no covariates

donor_latent_factors = M_F[:,1:]
treated_latent_factors = M_F[:,0]

# prepare targets and covariates
targets = []
covariates = []
for outcome in Outcomes:
    # take the first column
    targets.append(outcome[:, 0])
    # all but the first column
    covariates.append(outcome[:, 1:])

# run the learning algorithm
Q_final, w_final = lq.learnQ(targets, covariates, embedding_dim=5, n_iterations=2000, reg_Q=0, reg_w=0, verbose=False)

# Goal: should see M(Qw) = 2\mu_1 - for some reason its just mu_1!
print(donor_latent_factors @ (Q_final @ w_final))
print(treated_latent_factors)


tensor([-0.0085, -0.2089], dtype=torch.float64)
tensor([-0.0085, -0.2089], dtype=torch.float64)


Why does this happen? Its actually not that complicated, in this case. Here's my explanation:

SHOULD CALL THE DOT PRODUCT GAMMA

For notational simplicity, let $\mathbf{Q}\vec{w} = \vec{z}$. To minimize the loss, the goal is:
$$
\min_{\vec{w}}||\vec{y} - \mathbf{Y}(\mathbf{Q}\vec{w})|| = \min_{\vec{z}}||\vec{y} - \mathbf{Y}\vec{z}||,
$$ where $\mathbf{Y}$ represent the outcomes for the donors, and $\vec{y}$ are the outcomes for the treated unit. Note that since we are using only 1 timepoint, we can drop the subscript $t$. By hypothesis, we have the following:
$$
\vec{y} = \Lambda \vec{\mu_1}\quad\text{and}\quad \mathbf{Y} = \Lambda\mathbf{M},
$$ 
where $\vec{\mu_1}$ is the first column of the matrix $M_F$, which are the latent factors for the treated unit. Writing out the objective, we have:
$$
\min_{\vec{z}}||\Lambda \vec{\mu}_1 - \Lambda M \vec{z}|| = \min_{\vec{z}}||\Lambda (\vec{\mu}_1 - M \vec{z})|| = \min_{\vec{z}} [\Lambda (\vec{\mu} - \mathbf{M}\vec{z})]^T [\Lambda (\vec{\mu} - \mathbf{M}\vec{z})],
$$ 
which can be expressed as:
$$
\min_{\vec{z}} (\vec{\mu}_1 - \mathbf{M}\vec{z})^T(\Lambda^T\Lambda) (\vec{\mu}_1 - \mathbf{M}\vec{z}).
$$
Since $\Lambda^T\Lambda$ is a constant, the minimum occurs when $\vec{\mu}_1 =\mathbf{M}\vec{z}$. In otherwords, 
$$
\vec{\mu}_1 = (\mathbf{M}\mathbf{Q})\vec{w},
$$ 
meaning that $Q$ learns a representation of the latent factors of the donor units which contains the treated units latent factors in its convex hull. The question now is how this generalizes. First, we consider two timepoints.

In [ ]:
torch.manual_seed(42)

# settings
T = 1
N = 7
m = 2
f = 2
p = 2
# Generating data with covariates.
with_covariates = True

# we use Z_F now because we are passing covariates
Outcomes, latent_terms, M_F, Z_F = sim_Data(T, N, m, p, f, with_covariates)

# the latent factors, we will test to see if the method recovers these
donor_latent_factors = M_F[:,1:]
treated_latent_factors = M_F[:,0]

# Combining the outcomes with the covariates, so they can be passed together
Outcomes_tensor = Outcomes[0].detach().clone()
print("Outcomes:\n", Outcomes_tensor)
Z_F_tensor = Z_F.detach().clone()
print("Z_F:\n", Z_F_tensor)
stacked = torch.cat([Outcomes_tensor, Z_F_tensor], dim = 0)
print("Passed to model:\n", stacked)

# prepare targets and covariates
targets = []
covariates = []
for outcome in stacked.unsqueeze(0):
    # take the first column
    targets.append(outcome[:, 0])
    # all but the first column
    covariates.append(outcome[:, 1:])

print("Targets:\n", targets)
print("Covariates:\n", covariates)

# run the learning algorithm
Q_final, w_final = lq.learnQ(targets, covariates, embedding_dim=5, n_iterations=2000, reg_Q=0, reg_w=1, verbose=True)

# Hypothesis is that gamma times the donor latent factors will equal the latent factors for the control - THIS IS TRUE WHEN THE NUMBER OF DONORS IS LARGER THAN NUMBER OF COVARIATES?
# Gamma times donor latent factors:
print("M@gamma: \n", donor_latent_factors @ (Q_final @ w_final))
print("mu_1:\n", treated_latent_factors)


Outcomes:
 tensor([[ 0.5960, -0.4351, -0.5848, -1.2427,  1.6033,  1.3606, -0.2782],
        [ 0.6076,  2.1133, -0.5050,  1.5537, -1.8498, -1.3404,  0.1443]],
       dtype=torch.float64)
Z_F:
 tensor([[ 1.3221,  0.8172, -0.7658, -0.7506,  1.3525,  0.6863, -0.3278],
        [ 0.7950,  0.2815,  0.0562,  0.5227, -0.2384, -0.0499,  0.5263]],
       dtype=torch.float64)
Passed to model:
 tensor([[ 0.5960, -0.4351, -0.5848, -1.2427,  1.6033,  1.3606, -0.2782],
        [ 0.6076,  2.1133, -0.5050,  1.5537, -1.8498, -1.3404,  0.1443],
        [ 1.3221,  0.8172, -0.7658, -0.7506,  1.3525,  0.6863, -0.3278],
        [ 0.7950,  0.2815,  0.0562,  0.5227, -0.2384, -0.0499,  0.5263]],
       dtype=torch.float64)
Targets:
 [tensor([0.5960, 0.6076, 1.3221, 0.7950], dtype=torch.float64)]
Covariates:
 [tensor([[-0.4351, -0.5848, -1.2427,  1.6033,  1.3606, -0.2782],
        [ 2.1133, -0.5050,  1.5537, -1.8498, -1.3404,  0.1443],
        [ 0.8172, -0.7658, -0.7506,  1.3525,  0.6863, -0.3278],
        [ 0.28

We have shown: with 1 time point, and data generated under latent factor model 
$$
\mathbf{Y}_t = \boldsymbol{\Theta}_t\mathbf{Z}_F + \boldsymbol{\Lambda}_t \mathbf{M}_F,
$$ 
the procedure learns a matrix $\mathbf{Q}$ that perfectly recovers the latent factors $\mathbf{M}$. The important question: Does this still hold, given the same data, when we do not pass the covariates? 

In [ ]:
# USING THE SAME SETTINGS AS ABOVE, ONLY DIFFERENCE IS NOT PASSING COVARIATES.
torch.manual_seed(42)

# the latent factors, we will test to see if the method recovers these
donor_latent_factors = M_F[:,1:]
treated_latent_factors = M_F[:,0]

# Not passing covariates, just outcomes - data is generated with covariates, so this should be harder!
Outcomes_tensor = Outcomes[0].detach().clone()

# prepare targets and covariates
targets = []
covariates = []
for outcome in Outcomes_tensor.unsqueeze(0):
    # take the first column
    targets.append(outcome[:, 0])
    # all but the first column
    covariates.append(outcome[:, 1:])

# run the learning algorithm
Q_final, w_final = lq.learnQ(targets, covariates, embedding_dim=5, n_iterations=2000, reg_Q=0, reg_w=1, verbose=False)

# Gamma times donor latent factors:
print("M@gamma: \n", donor_latent_factors @ (Q_final @ w_final))
print("mu_1:\n", treated_latent_factors)

M@gamma: 
 tensor([-0.0404, -0.6716], dtype=torch.float64)
mu_1:
 tensor([-0.0085, -0.2089], dtype=torch.float64)


The answer: NO. Without passing the covariates, no matter how many donors we use, the latent factors are not recovered. Now, how does this work with multiple timepoints? 

In [ ]:
# The final loss can be large while this mean is small. What does that indicate?
# try passing the covariates also. Feels like given this info it should be able to sus out the latent factors.


diff = [((A.numpy().T @ A.numpy() @ (Q_final @ w_final)) - (A.numpy().T @ d.numpy())) for A, d in zip(covariates, targets)]
print(diff)
print(np.mean(diff))